## Study notes for ViT

**Author: Haozhe Jia**\
This is a study note for building ViT from scratch to understand the architecture
and mechanism of ViT.

`TODO: add related papers references`

> **On this notebook and the `vit/` package.** The modules below are built from
> scratch for teaching, but they are kept **identical** to the extracted package in
> [`vit/`](../vit/), which is the source of truth used by `train.py`. The last
> section of this notebook asserts that the two agree, so this notebook cannot
> silently drift away from the code that actually trains.

### The shape roadmap

The whole architecture is one chain of shape transformations. Each section below
is exactly one arrow:

| Stage | Shape |
| --- | --- |
| input image | `(B, C, H, W)` |
| `Conv2d` patchify | `(B, d_model, P_row, P_col)` |
| flatten + transpose | `(B, P, d_model)` |
| prepend `[CLS]` + add position | `(B, P+1, d_model)` |
| encoder stack (shape-preserving) | `(B, P+1, d_model)` |
| take `[CLS]`, LayerNorm | `(B, d_model)` |
| classifier head | `(B, n_classes)` |

If you follow only the shapes, you still understand ViT.

In [ ]:
import sys
sys.path.insert(0, "..")   # so `import vit` resolves from notebooks/

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets.mnist import MNIST

torch.manual_seed(0)

In [ ]:
def load_image(path, image_size=224):
    """Load an image file -> (1, 3, image_size, image_size) float tensor in [0, 1]."""
    img = Image.open(path).convert("RGB")
    transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
    ])
    x = transform(img).unsqueeze(0)  # add batch dim
    return x


def show_patches(x, patch_size=16):
    """BEFORE view: the original image, and the image cut into the patches
    that PatchEmbedding's Conv2d will see."""
    img = x[0]                                      # (3, H, W)
    n = img.shape[1] // patch_size                  # patches per side, e.g. 14

    # (3, H, W) -> (3, n, n, patch_size, patch_size)
    patches = img.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)

    fig = plt.figure(figsize=(9, 4.5))

    ax = fig.add_subplot(1, 2, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title("Original image")
    ax.axis("off")

    # right half: n x n grid of separated tiles
    for row in range(n):
        for col in range(n):
            ax = fig.add_subplot(n, 2 * n, row * 2 * n + n + col + 1)
            ax.imshow(patches[:, row, col].permute(1, 2, 0))
            ax.axis("off")

    fig.suptitle(f"{n}x{n} = {n * n} patches of {patch_size}x{patch_size} pixels")
    plt.tight_layout()
    plt.show()


def show_embeddings(x, patch_embed, use_pca=False):
    """AFTER view: run PatchEmbedding and visualize the output sequence."""
    with torch.no_grad():
        out = patch_embed(x)                        # (1, num_patches, d_model)

    print(f"input : {tuple(x.shape)}")
    print(f"output: {tuple(out.shape)}  <- sequence of {out.shape[1]} tokens, "
          f"{out.shape[2]} dims each")

    tokens = out[0]                                 # (num_patches, d_model)
    n = int(tokens.shape[0] ** 0.5)                 # e.g. 14

    fig, axes = plt.subplots(1, 2 if use_pca else 1, figsize=(10, 4), squeeze=False)

    ax = axes[0, 0]
    im = ax.imshow(tokens, aspect="auto", cmap="viridis")
    ax.set_title("Patch embeddings (one row = one token)")
    ax.set_xlabel("hidden dim")
    ax.set_ylabel("patch index")
    fig.colorbar(im, ax=ax)

    if use_pca:
        # project d_model dims -> 3, reshape back to the n x n grid, show as RGB
        _, _, v = torch.pca_lowrank(tokens, q=3)
        rgb = tokens @ v                            # (num_patches, 3)
        rgb = (rgb - rgb.min(0).values) / (rgb.max(0).values - rgb.min(0).values + 1e-8)
        ax = axes[0, 1]
        ax.imshow(rgb.reshape(n, n, 3))
        ax.set_title("Same tokens, PCA to RGB on the patch grid")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return out

# 1. Patch embedding

A transformer eats a *sequence of vectors*. An image is a grid of pixels. Patch
embedding is the adapter between the two: cut the image into non-overlapping
`patch_size` tiles, flatten each tile, and linearly project it to `d_model`.

The implementation is a `Conv2d` with `kernel_size == stride == patch_size`. This
is not a convolutional approximation of the paper's "flatten and multiply by a
matrix" — with the stride equal to the kernel, the conv windows tile the image
without overlap, and each window's dot product with the kernel **is** that linear
projection. We prove this numerically two cells down.

Parameter count: `d_model * n_channels * patch_size^2` weights + `d_model` bias.

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self, d_model, img_size, patch_size, n_channels):
    super().__init__()
    self.d_model = d_model # Dimensionality of Model
    self.img_size = img_size # Image Size
    self.patch_size = patch_size # Patch Size
    self.n_channels = n_channels # Number of Channels

    """
    Learnable parameter nn.Conv2d which register its weight and bias as 'nn.Parameter',
    Parameter count: d_model * n_channels * patch_size² + d_model
    weight: shape(d_model, n_channels, patch_size, patch_size)
      It flattening each patch into a vec of length `n_channels * patch_size^2` and multiplying by a learned (d_model, n_channels * patch_size ** 2) matrix
    bias: shape(d_model,)
    """
    self.linear_project = nn.Conv2d(
      self.n_channels,
      self.d_model,
      kernel_size=self.patch_size,
      stride=self.patch_size)

  # B: Batch Size
  # C: Image Channels
  # H: Image Height
  # W: Image Width
  # P_row: Patch Row = H / patch_size
  # P_col: Patch Column = W / patch_size
  # P = Patch numbers = P_row * P_col

  def forward(self, x):
    x = self.linear_project(x) # (B, C, H, W) -> (B, d_model, P_row, P_col)
    '''
    Flatten everything from dim 2 onward into one dim,
    so two spatial dims (P_row, P_col) merge into a single P = P_row × P_col
    The patch at grid position (i, j) lands at sequence index i * P_col + j
    '''
    x = x.flatten(2) # (B, d_model, P_row, P_col) -> (B, d_model, P)

    x = x.transpose(1, 2) # (B, d_model, P) -> (B, P, d_model)

    return x

### The `Conv2d` trick is exact

Claim: a strided conv equals "unfold each patch into a vector, then apply one
`Linear`". If that is true, `conv.weight.view(d_model, -1)` is literally the
projection matrix from the paper, and `nn.Unfold` + `Linear` must reproduce the
conv output bit for bit.

In [ ]:
patch_embed = PatchEmbedding(d_model=8, img_size=(32, 32), patch_size=(16, 16), n_channels=3)
x = torch.randn(2, 3, 32, 32)

# Route 1: the conv, as implemented above.
conv_out = patch_embed(x)                                   # (B, P, d_model)

# Route 2: the paper's description, done literally.
W = patch_embed.linear_project.weight.view(8, -1)           # (d_model, C*ph*pw)
b = patch_embed.linear_project.bias                         # (d_model,)
patches = F.unfold(x, kernel_size=(16, 16), stride=(16, 16))  # (B, C*ph*pw, P)
manual_out = (W @ patches + b[None, :, None]).transpose(1, 2)  # (B, P, d_model)

print("conv  :", tuple(conv_out.shape))
print("manual:", tuple(manual_out.shape))
print("max abs diff:", (conv_out - manual_out).abs().max().item())
assert torch.allclose(conv_out, manual_out, atol=1e-5)
print("\nIdentical -> the Conv2d IS the patch-flattening linear projection.")

In [ ]:
# See it on a real image: 224/16 = 14 -> 14x14 = 196 tokens.
x_img = load_image("../../assets/ave-mujica-resized.jpeg", image_size=224)
show_patches(x_img, patch_size=16)

demo_embed = PatchEmbedding(d_model=768, img_size=(224, 224), patch_size=(16, 16), n_channels=3)
_ = show_embeddings(x_img, demo_embed, use_pca=True)

# 2. Positional encoding

Two things happen here, and it is worth separating them.

**The `[CLS]` token.** A single learnable vector prepended to the sequence. It
corresponds to no patch, so it carries no image content of its own — whatever it
holds at the output, it got by *attending to* the patches. That is exactly why it
is the thing we classify from, and why its attention row is the thing you
visualize.

**The position embedding.** Self-attention is permutation-invariant: it sees a
*set* of tokens, not a sequence. Shuffle the patches and, without position
information, the model literally cannot tell. So we add a position vector to each
token.

We use a **learnable** `nn.Parameter` of shape `(1, max_seq_length, d_model)` —
this is what ViT does, and it is a real departure from the sinusoidal table of
"Attention Is All You Need" (kept below as a commented reference). The `use_pe`
flag exists so the ablation is one config change away: set it to `False` and
watch accuracy fall.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length, use_pe=True):
        super().__init__()
        # Learnable parameter : CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model)) # (1, 1, d_model)
        # Learnable parameter Positional Encodings
        self.use_pe = use_pe
        if use_pe:
            self.pe = nn.Parameter(torch.randn(1, max_seq_length, d_model))
        """
        # Creating positional encodings
        pe = torch.zeros(max_seq_length, d_model) # (max_seq_length, d_model)
        # Sinusoidal Positional Embeddings
        for pos in range(max_seq_length):
            for i in range(d_model):
                if i % 2 == 0:
                    pe[pos][i] = np.sin( pos / ( 10000 ** ( i / d_model )) )
                else:
                    pe[pos][i] = np.cos( pos / 10000 ** ( (i - 1) / d_model) )
        self.register_buffer('pe', pe.unsqueeze(0)) # (1, max_seq_length, d_model)
        """

    def forward(self, x): # x: (B, n_patches, d_model)
        # Expand to have class token for every image in batch, 1 in expand() means "keep this dimension as-is."
        tokens_batch = self.cls_token.expand(x.size()[0], -1, -1) # (1, 1, d_model) -> (B, 1, d_model)

        # Add class tokens to the beginning of each patch embedding
        # Input: (B, 1, d_model) ⧺ (B, n_patches, d_model) along dim 1
        x = torch.cat((tokens_batch, x),dim=1) # -> (B, n_patches + 1, d_model)

        # Adding positional encoding to embeds
        # Input x: (B, n_patches + 1, d_model), self.pe: (1, max_seq_length, d_model)
        # Skipped entirely when use_pe=False: `self.pe` does not exist in that
        # case, and self-attention is then permutation-invariant over patches.
        if self.use_pe:
            x = x + self.pe # -> (B, max_token_length + 1, d_model)

        return x

In [ ]:
pe = PositionalEncoding(d_model=8, max_seq_length=4 + 1)
tokens = torch.randn(2, 4, 8)          # 4 patches
out = pe(tokens)
print(f"{tuple(tokens.shape)} -> {tuple(out.shape)}   (+1 token: the CLS)")

# Why position matters: without PE, shuffling the patches changes nothing.
no_pe = PositionalEncoding(d_model=8, max_seq_length=5, use_pe=False)
shuffled = tokens[:, torch.randperm(4)]
same = torch.allclose(no_pe(tokens)[:, 1:].sort(dim=1).values,
                      no_pe(shuffled)[:, 1:].sort(dim=1).values)
print("use_pe=False -> patch set is identical under shuffling:", same)

# 3. Transformer encoder

Standard pre-LayerNorm encoder blocks. Three things worth noticing:

**`is_causal=False`.** This is the single line separating vision from language.
In [jimmy-gpt2](../../Jimmy-gpt2/) the same attention runs with a causal mask,
because a language model must not see the future. An image has no "future" — every
patch may look at every other patch — so the mask is dropped and attention is fully
bidirectional. The rest of the module is the same code.

**`F.scaled_dot_product_attention`.** PyTorch picks a backend from the dtype and
shapes. FlashAttention requires fp16/bf16, so these fp32 configs never get it: the
MNIST config (head size 3) falls all the way back to the **math** backend, and
Tiny ImageNet (head size 64) gets the memory-efficient one.

Either way the attention map is unavailable — and the reason is the **API**, not
fusion. `scaled_dot_product_attention` returns exactly one tensor, its output. No
backend returns the `(B, n_heads, T, T)` probabilities, and forcing the math
backend does not help: it builds the matrix internally and drops it before you can
reach it. To see the map you must compute it yourself, which is what the
`store_attn` toggle below does.

**`DropPath` (stochastic depth).** Drops an entire residual branch for some samples
in the batch, so they pass through the skip connection alone. The rate scales
linearly from 0 at the first block to `drop_path` at the last (the DeiT/timm
schedule, applied in `ViTBackbone`).

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Learnable parameter count in MultiHeadAttention:
    c_attn: 3 * d_model· * model + 3 * d_model = 3 * d_model * (d_model + 1)
    c_proj: d_model * d_model + d_model     = d_model * (d_model + 1)
    Total:  4 * d_model * (d_model + 1)
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        # key, query, value projection/weights for all heads, but in a batch
        self.c_attn = nn.Linear(d_model, 3 * d_model)
        # output projection
        self.c_proj = nn.Linear(d_model, d_model)        # regularization
        self.n_head = n_heads
        self.d_model = d_model

        # Attention-map capture, off by default so training never pays for it.
        # `F.scaled_dot_product_attention` returns only its output tensor: no
        # backend hands back the (B, nh, T, T) probabilities, so the map cannot
        # be recovered from the fast path at all -- not even by forcing the MATH
        # backend, which builds the matrix internally and then drops it. Setting
        # `store_attn = True` swaps in the eager path below, which is the same
        # math unfused, and keeps a reference to the matrix on `attn_map`.
        self.store_attn = False
        self.attn_map = None   # (B, nh, T, T) after a forward with store_attn=True

    def _eager_attention(self, q, k, v):
        """Unfused attention. Identical math to F.scaled_dot_product_attention
        (same 1/sqrt(head_size) scale, no mask), but the probability matrix is
        materialized so it can be stashed for visualization."""
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1)) # (B, nh, T, T)
        att = att.softmax(dim=-1)
        # Detached: this is a diagnostic, it must never hold the graph alive.
        self.attn_map = att.detach()
        return att @ v # (B, nh, T, hs)

    def forward(self, x):
        B, T, C = x.size() # Batch size, sequence length, embedding dimensionality (n_embd)
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        q = q.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        v = v.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        if self.store_attn:
            y = self._eager_attention(q, k, v)
        else:
            # PyTorch picks the backend from dtype/shape: FlashAttention needs
            # fp16/bf16, so these fp32 configs get mem-efficient or math.
            y = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        y = y.transpose(1,2).contiguous().view(B,T,C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y

class FFN(nn.Module):
    """
    Learnable parameter count in FFN:
    c_fc: r * d_model * d_model + r * d_model
    c_proj: r * d_model * d_model + d_model
    Total:  2 * r * d_model * d_model + (r + 1)·d_model
    """
    def __init__(self, d_model, r_ffn):
        super().__init__()
        self.c_fc = nn.Linear(d_model, r_ffn * d_model)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(r_ffn * d_model, d_model)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class DropPath(nn.Module):
    """
    Stochastic depth: it drops an entire residual branch for some samples in the batch.

    During training some samples get x = x + attn(x) and others just get x = x,
    The whole attention (or MLP) computation is zeroed out for that sample,
    so it passes through the skip connection only.
    """
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        # one Bernoulli value per sample, broadcast over the other dims
        shape = (x.shape[0],) + (1,) * (x.ndim - 1) # For input (B,P,d_model), create (B, 1, 1)
        mask = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        mask.floor_()
        return x / keep_prob * mask

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, r_ffn=4, drop_path=0.0):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads

        # Sub-Layer 1 Normalization
        self.ln_1 = nn.LayerNorm(d_model)
        # Multi-Head Attention
        self.attn = MultiHeadAttention(d_model, n_heads)
        # Sub-Layer 2 Normalization
        self.ln_2 = nn.LayerNorm(d_model)
        # FFN(MLP) layer
        self.ffn = FFN(d_model, r_ffn)
        # Stochastic depth on each residual branch
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        # Residual Connection After Sub-Layer 1
        x = x + self.drop_path(self.attn(self.ln_1(x)))
        # Residual Connection After Sub-Layer 2
        x = x + self.drop_path(self.ffn(self.ln_2(x)))
        return x

In [ ]:
block = TransformerEncoder(d_model=8, n_heads=2)
z = torch.randn(2, 5, 8)
print(f"encoder block: {tuple(z.shape)} -> {tuple(block(z).shape)}   (shape-preserving)")

# Check the documented parameter count: 4 * d_model * (d_model + 1)
mha = MultiHeadAttention(d_model=8, n_heads=2)
print("MHA params:", sum(p.numel() for p in mha.parameters()), "== 4*8*(8+1) =", 4 * 8 * (8 + 1))

### Which backend actually runs?

Worth checking rather than assuming. FlashAttention requires fp16/bf16 inputs, so
an fp32 model never gets it — regardless of what the code comment says.

In [ ]:
from torch.nn.attention import SDPBackend, sdpa_kernel

def usable_backends(nh, T, hs, dtype=torch.float32):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    q = torch.randn(1, nh, T, hs, device=dev, dtype=dtype)
    ok = []
    for name, be in [("FLASH", SDPBackend.FLASH_ATTENTION),
                     ("MEM_EFFICIENT", SDPBackend.EFFICIENT_ATTENTION),
                     ("MATH", SDPBackend.MATH)]:
        try:
            with sdpa_kernel(be):
                F.scaled_dot_product_attention(q, q, q)
            ok.append(name)
        except Exception:
            pass
    return ok

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print("mnist   (nh=3, T=5,  hs=3,  fp32):", usable_backends(3, 5, 3))
    print("tiny-in (nh=3, T=65, hs=64, fp32):", usable_backends(3, 65, 64))
    if torch.cuda.is_available():
        print("tiny-in (nh=3, T=65, hs=64, fp16):", usable_backends(3, 65, 64, torch.float16))

### Getting the attention map out

None of those backends will give you the `(B, n_heads, T, T)` matrix: SDPA returns
one tensor, its output. So `store_attn` swaps in an eager re-implementation that
keeps the matrix. The test that matters is that it changes *nothing* about the
model — same weights, same math, same answer.

In [ ]:
mha = MultiHeadAttention(d_model=8, n_heads=2).eval()
z = torch.randn(2, 5, 8)

with torch.no_grad():
    fast = mha(z)                 # default: SDPA
print("attn_map with store_attn=False:", mha.attn_map)

mha.store_attn = True
with torch.no_grad():
    eager = mha(z)                # eager path, map retained

print("\nlogits max abs diff (fast vs eager):", (fast - eager).abs().max().item())
assert torch.allclose(fast, eager, atol=1e-5), "eager path diverges from SDPA!"

a = mha.attn_map
print("attn_map shape:", tuple(a.shape), "  (B, n_heads, T, T)")
print("every row sums to 1:", torch.allclose(a.sum(-1), torch.ones_like(a.sum(-1))))
print("detached (no graph held):", not a.requires_grad)
print("\nCLS row, head 0 -> how much the CLS token attends to each token:")
print(" ", a[0, 0, 0].tolist())

# 4. Vision Transformer

`ViTBackbone` runs embeddings -> encoder stack -> final LayerNorm and returns the
`[CLS]` token: a single `(B, d_model)` vector per image. `ViTClassifier` puts one
`nn.Linear` on top.

The split matters — the backbone is the reusable representation, the head is the
task. Swap the head and the same backbone does a different job.

**The head returns raw logits, with no `Softmax`.** `nn.CrossEntropyLoss` applies
`log_softmax` internally, so adding a `Softmax` in the model would apply it twice:
the gradients get squashed and training is quietly crippled. This is a genuinely
easy mistake to make, and an earlier version of this notebook made it.

In [ ]:
class ViTBackbone(nn.Module):
    def __init__(self, d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers, use_pe=True, r_ffn=4, drop_path=0.0):
        super().__init__()
        assert img_size[0] % patch_size[0] == 0 and img_size[1] % patch_size[1] == 0, "img size dim must be divisible by patch_size dim"

        self.d_model = d_model # Dimensionality of model
        self.n_classes = n_classes # Number of classes
        self.img_size = img_size # Image size
        self.patch_size = patch_size # Patch size
        self.n_channels = n_channels # Number of channels
        self.n_heads = n_heads # Number of attention heads
        self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1]) # Number of patches
        self.max_seq_length = 1 + self.n_patches # number of patches + [CLS] token
        self.patch_embedding = PatchEmbedding(self.d_model, self.img_size, self.patch_size, n_channels)
        self.positional_encoding = PositionalEncoding(self.d_model, self.max_seq_length, use_pe=use_pe)

        # Linearly increasing stochastic-depth rate: 0 at the first block up to
        # `drop_path` at the last (the standard DeiT/timm schedule).
        dpr = [drop_path * i / max(1, n_layers - 1) for i in range(n_layers)]
        # We create number of n_layers of transformer stack, with increasing dropout rate
        self.transformer_encoders = nn.Sequential(
            *[TransformerEncoder(
                self.d_model, self.n_heads, r_ffn, dpr[i]
            ) for i in range(n_layers)])
        # Initialize LayerNorm variable for last layer.
        self.ln_f = nn.LayerNorm(d_model)

    def forward(self, images):
        x = self.patch_embedding(images) # (B, P, dim_model)
        x = self.positional_encoding(x)  # (B, max_token_length + 1, d_model)
        x = self.transformer_encoders(x)  # (B, max_token_length + 1, d_model)
        return self.ln_f(x[:, 0]) # Return normalized cls token (B, d_model)

class ViTClassifier(nn.Module):
    def __init__(self, backbone, n_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.d_model, n_classes)

    def forward(self, images):
        return self.head(self.backbone(images))

### Parity check against `vit/`

The cell below builds the same architecture from the notebook classes and from the
installed package, and compares parameter counts and output shapes. If someone
edits `vit/` without updating this notebook (or vice versa), this cell fails —
which is the point.

In [ ]:
import vit

cfg = dict(d_model=9, n_classes=10, img_size=(32, 32), patch_size=(16, 16),
           n_channels=1, n_heads=3, n_layers=3)

mine = ViTClassifier(ViTBackbone(**cfg), cfg["n_classes"])
theirs = vit.ViTClassifier(vit.ViTBackbone(**cfg), cfg["n_classes"])

n_mine = sum(p.numel() for p in mine.parameters())
n_theirs = sum(p.numel() for p in theirs.parameters())
print(f"notebook params: {n_mine}")
print(f"vit/ params    : {n_theirs}")

probe = torch.randn(2, 1, 32, 32)
assert n_mine == n_theirs, "notebook has drifted from vit/"
assert mine(probe).shape == theirs(probe).shape == (2, 10)
print("\nNotebook and vit/ agree.")

# 5. Training and testing

The config below matches [`configs/mnist-baseline.yaml`](../configs/mnist-baseline.yaml),
which is what `train.py` uses. This loop is the stripped-down version: no
augmentation, no validation split, no scheduler — just enough to see it learn.
For the real recipe, use `python train.py --config configs/mnist-baseline.yaml`.

In [ ]:
d_model = 9
n_classes = 10
img_size = (32,32)
patch_size = (16,16)
n_channels = 1
n_heads = 3
n_layers = 3
batch_size = 128
epochs = 5
alpha = 0.005

transform = T.Compose([
  T.Resize(img_size),
  T.ToTensor()
])

train_set = MNIST(
  root="./../../datasets", train=True, download=True, transform=transform
)
test_set = MNIST(
  root="./../../datasets", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_set, shuffle=False, batch_size=batch_size)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device, f"({torch.cuda.get_device_name(device)})" if torch.cuda.is_available() else "")

transformer = ViTClassifier(
    ViTBackbone(d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers),
    n_classes
).to(device)

optimizer = Adam(transformer.parameters(), lr=alpha)
criterion = nn.CrossEntropyLoss()   # expects raw logits; the model has no Softmax

for epoch in range(epochs):

  training_loss = 0.0
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer.zero_grad()

    outputs = transformer(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    training_loss += loss.item()

  print(f'Epoch {epoch + 1}/{epochs} loss: {training_loss  / len(train_loader) :.3f}')

In [ ]:
correct = 0
total = 0

transformer.eval()
with torch.no_grad():
  for data in test_loader:
    images, labels = data
    images, labels = images.to(device), labels.to(device)

    outputs = transformer(images)

    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  print(f'\nModel Accuracy: {100 * correct // total} %')